In [1]:
from peft import LoraConfig , get_peft_model
import torch 
from transformers import Seq2SeqTrainer , Seq2SeqTrainingArguments , AutoModelForSeq2SeqLM
from torch.optim import AdamW #or any other that we need here 
from datasets import load_dataset
from transformers import AutoTokenizer



In [2]:
dataset = load_dataset("squad_v2")

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


In [4]:
print(dataset['train'][0])

{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}


In [5]:
for example in dataset['train']:
    if len(example['answers']['text']) == 0:
        print(example)
        break 

{'id': '5a8d7bf7df8bba001a0f9ab1', 'title': 'The_Legend_of_Zelda:_Twilight_Princess', 'context': 'The Legend of Zelda: Twilight Princess (Japanese: ゼルダの伝説 トワイライトプリンセス, Hepburn: Zeruda no Densetsu: Towairaito Purinsesu?) is an action-adventure game developed and published by Nintendo for the GameCube and Wii home video game consoles. It is the thirteenth installment in the The Legend of Zelda series. Originally planned for release on the GameCube in November 2005, Twilight Princess was delayed by Nintendo to allow its developers to refine the game, add more content, and port it to the Wii. The Wii version was released alongside the console in North America in November 2006, and in Japan, Europe, and Australia the following month. The GameCube version was released worldwide in December 2006.[b]', 'question': 'What category of game is Legend of Zelda: Australia Twilight?', 'answers': {'text': [], 'answer_start': []}}


In [6]:
def preprocess(example):
    input_text = f"question:{example['question']} context: {example['context']}"
    
    if len(example['answers']['text']) == 0:
        output_text = "unanswerable"
    
    else:
        output_text = example['answers']['text'][0]
    
    return {"input": input_text, "output": output_text}

In [7]:
preprocessed_data = dataset.map(preprocess) 

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [8]:
print(preprocessed_data)
print(preprocessed_data['train'][0]) 

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'input', 'output'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'input', 'output'],
        num_rows: 11873
    })
})
{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the B

In [9]:
unanswerable_train = sum(1 for ex in preprocessed_data['train'] if ex['output'] == 'unanswerable')
unanswerable_val = sum(1 for ex in preprocessed_data['validation'] if ex ['output'] == 'unanswerable')

print(f"Train unanswerable: {unanswerable_train} / {preprocessed_data['train'].num_rows}")
print(f"Val unanswerable: {unanswerable_val} / {preprocessed_data['validation'].num_rows}")

Train unanswerable: 43498 / 130319
Val unanswerable: 5945 / 11873


In [10]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base") 


def tokenize(example):
    x = tokenizer(example['input'], max_length = 512, truncation = True)
    y = tokenizer(example['output'], max_length = 128, truncation = True)
    
    return {
        "input_ids": x['input_ids'],
        "attention_mask": x['attention_mask'],
        "labels": y['input_ids'],
    }


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [11]:
tokenized_dataset = preprocessed_data.map(tokenize)

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [16]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11873
    })
})


In [12]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
lora_config = LoraConfig(
    r = 8, 
    lora_alpha = 16, 
    target_modules = ["q","v"], 
    lora_dropout = 0.1, 
    task_type = "SEQ_2_SEQ_LM"
)

In [14]:
model = get_peft_model(model, lora_config) 
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [46]:
training_args = Seq2SeqTrainingArguments(
    output_dir = ".flan-5g-lora-squad",
    num_train_epochs = 3, 
    per_device_train_batch_size= 4, 
    per_device_eval_batch_size= 4, 
    eval_strategy = "epoch", 
    save_strategy= "epoch", 
    learning_rate = 3e-4, 
    fp16 = True, 
    predict_with_generate= True, 
    logging_steps = 100, 
    gradient_accumulation_steps= 4, 
    report_to = "none"
)

In [47]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

In [61]:
small_train = tokenized_dataset['train'].shuffle(seed=42).select(range(30000))
small_val = tokenized_dataset['validation'].shuffle(seed=42).select(range(3000))

In [62]:
trainer = Seq2SeqTrainer(
    model= model,
    args= training_args,
    train_dataset= small_train,
    eval_dataset= small_val,
    processing_class = tokenizer,
    data_collator = data_collator
)

In [63]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,46.518550,8.916349
2,44.712183,8.673770
3,44.589756,8.609550


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=2814, training_loss=45.57812693039407, metrics={'train_runtime': 6451.8805, 'train_samples_per_second': 13.949, 'train_steps_per_second': 0.436, 'total_flos': 3.776842997858304e+16, 'train_loss': 45.57812693039407, 'epoch': 3.0})

In [51]:
!nvidia-smi

Mon May 11 13:17:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             28W /   70W |    9581MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [64]:
input_text = "question: What is Ayushman Bharat? context: Ayushman Bharat is a health insurance scheme launched by the Government of India in 2018. It provides health coverage of up to 5 lakh rupees per family per year for secondary and tertiary care hospitalization."


In [65]:
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

unanswerable


In [68]:
input_text = "question: What is the capital of France? context: France is a country in Europe. Its capital city is Paris."
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Paris


In [67]:
from collections import Counter
answers = [ex['answers']['text'] for ex in dataset['train'].select(range(30000))]
empty = sum(1 for a in answers if len(a) == 0)
print(f"Unanswerable: {empty}, Answerable: {30000 - empty}")


Unanswerable: 6954, Answerable: 23046


In [57]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
base_model.eval()

inputs = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


2018


In [30]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_available())
print(torch.cuda.device_count())

Tesla T4
True
2
